# 2. Clusters, PCA, and Correlations

**Pipeline position:** runs after `1_data_prep.ipynb`. Reads `panel.parquet`
and re-fits the k=5 country clustering in-notebook (instead of importing
labels from the upstream repo). Then computes correlations, PCA, the
rent-share breakdown by cluster, and the median ECI trajectory.

## What this notebook does

1. **K-means (k=5) on 1995 cross-section** — fits on the four rent shares
   (oil, gas, mineral, forestry, all as % of GDP) plus log production per
   capita. Standardised features, `random_state=42`. Cluster *names* are
   not hard-coded: they are derived from each cluster's centroid by
   inspecting which rent share dominates and the production level. Names
   follow the data instead of being a stale mapping from an earlier fit.
2. **Rent breakdown by cluster** — table of mean rent % of GDP by
   resource type (oil, gas, mineral, forestry, total) for each cluster.
   This is the ground truth for the cluster names: if cluster names look
   wrong, this table shows why.
3. **Pearson correlations with ECI** — feature-by-feature on the panel.
4. **PCA scatter and loadings** — PC1/PC2 from the 1995 snapshot, using
   the wider macro feature set (not the rent-only clustering features).
5. **Median ECI by cluster, over time** — pre-aggregated for the
   trajectory chart.

## Outputs

- `clusters_1995.csv` — Country Code, Country, ClusterLabels (overwrites
  the file from notebook 1)
- `cluster_rent_breakdown.csv` — cluster, n_countries, mean rent % of GDP
  per resource type, total NR, mean log production per capita
- `corr_with_eci.csv` — feature, short label, category, Pearson r
- `pca_scatter.csv` — country, cluster, PC1, PC2
- `pca_loadings.csv` — feature, short label, PC1, PC2, variance explained
- `median_eci_by_cluster.csv` — year, cluster, median ECI, SE, count

## Note on the panel column

The `ClusterLabels` column already in `panel.parquet` (merged in by
notebook 1 from the upstream `clusters1995.csv`) is overwritten by the
new assignments before the median-ECI aggregation runs. To make this
permanent for downstream notebooks (3, 4, 5 and the viz notebooks), the
panel is re-saved with the new labels.


In [ ]:
import os, sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from _config import (ARTIFACTS, ECI_COL, CAT_MAP, NAME_MAP,
                     artifact_path)

panel = pd.read_parquet(artifact_path('panel.parquet'))
print(f'Panel: {panel["Country Code"].nunique()} countries, {len(panel):,} obs')


## K-means clustering (k=5) on the 1995 cross-section

Four features: oil rents, gas rents, mineral rents (all as % of GDP) and
log production per capita. Standardised to zero mean and unit variance
before fitting so no feature dominates by scale alone.

**Forestry rents are deliberately excluded from the clustering features.**
The headline sample is forest-adjusted (forestry subtracted from total NR
rents before applying the 1% threshold), so including forestry rents as
a clustering input would reintroduce the same bias the sample selection
corrects for. Forestry is still reported in the breakdown table further
down for diagnostic purposes — but it does not influence which cluster
a country ends up in.

Missing rent values are filled with zero before fitting (a country missing
oil rent data is treated as having zero oil rent in 1995). Same convention
notebook 5 uses for the rent-decomposition charts.


In [ ]:
# All four rent components — used by the breakdown table further down.
RENT_COLS = [
    'Oil rents (% of GDP)',
    'Natural gas rents (% of GDP)',
    'Mineral rents (% of GDP)',
    'Forestry rents (% of GDP)',
]
# Subset used by K-means: forestry deliberately excluded (see markdown above).
CLUSTER_RENT_COLS = [
    'Oil rents (% of GDP)',
    'Natural gas rents (% of GDP)',
    'Mineral rents (% of GDP)',
]
PROD_COL = 'Total_Production_Value_Per_Capita'

snap = panel[panel['Year'] == 1995].copy()
for c in RENT_COLS:
    snap[c] = snap[c].fillna(0)
snap['log_prod_pc'] = np.log1p(snap[PROD_COL].fillna(0))

cluster_feats = CLUSTER_RENT_COLS + ['log_prod_pc']
snap_fit = snap.dropna(subset=cluster_feats).copy()

scaler = StandardScaler()
X_clust = scaler.fit_transform(snap_fit[cluster_feats].values)

km = KMeans(n_clusters=5, random_state=42, n_init=20).fit(X_clust)
snap_fit['cluster_id'] = km.labels_
print(f'Fit on {len(snap_fit)} countries using {len(cluster_feats)} features:')
for f in cluster_feats:
    print(f'  - {f}')
print(f'Cluster sizes: {dict(pd.Series(km.labels_).value_counts().sort_index())}')


### Auto-assign cluster names from centroids

Each cluster's centroid (in standardised feature space) is unscaled back
to original % of GDP units. Names are then assigned by which rent
component dominates and how high production per capita is:

- Oil-dominant + highest log_prod_pc → `Petrostates`
- Oil-dominant, lower log_prod_pc → `Oil Exporters`
- Mineral-dominant → `Mining Exporters`
- Gas-dominant → `Gas Exporters`
- Residual cluster → `Diversified Producers`

There is no Forestry Intensive label because forestry is not a clustering
input. If multiple clusters share the same dominant rent (which can happen
with k=5 if oil heterogeneity is large), the second one falls through to
the residual `Diversified Producers` name. The centroid table is printed
for verification.


In [ ]:
centroids_orig = scaler.inverse_transform(km.cluster_centers_)
centroids_df = pd.DataFrame(centroids_orig, columns=cluster_feats)
centroids_df.index.name = 'cluster_id'
centroids_df['n'] = pd.Series(km.labels_).value_counts().sort_index().values

print('Cluster centroids (mean values, original units):')
print(centroids_df.round(2).to_string())
print()

# Dominance is computed over the three rent features the clustering used.
# Forestry is not eligible to be 'dominant' because it was not an input.
centroids_df['dominant_rent'] = centroids_df[CLUSTER_RENT_COLS].idxmax(axis=1)

name_map = {}
claimed = set()

# 1. Oil-dominant: split into Petrostates (highest prod) and Oil Exporters
oil_dom = centroids_df[centroids_df['dominant_rent'] == 'Oil rents (% of GDP)']
if len(oil_dom) >= 1:
    petro = oil_dom.sort_values('log_prod_pc', ascending=False).index[0]
    name_map[petro] = 'Petrostates'
    claimed.add(petro)
    if len(oil_dom) >= 2:
        oil_exp = oil_dom.sort_values('log_prod_pc', ascending=False).index[1]
        name_map[oil_exp] = 'Oil Exporters'
        claimed.add(oil_exp)

# 2. Mineral-dominant
for cid, row in centroids_df.iterrows():
    if cid in claimed:
        continue
    if row['dominant_rent'] == 'Mineral rents (% of GDP)':
        name_map[cid] = 'Mining Exporters'
        claimed.add(cid)
        break

# 3. Gas-dominant
for cid, row in centroids_df.iterrows():
    if cid in claimed:
        continue
    if row['dominant_rent'] == 'Natural gas rents (% of GDP)':
        name_map[cid] = 'Gas Exporters'
        claimed.add(cid)
        break

# 4. Whatever remains: Diversified Producers (lowest rent intensity overall,
#    typically the residual catch-all cluster). If more than one cluster
#    falls through, the highest-production one keeps the canonical name and
#    the others are numbered.
remaining = [c for c in centroids_df.index if c not in claimed]
if len(remaining) == 1:
    name_map[remaining[0]] = 'Diversified Producers'
else:
    rem_sorted = (centroids_df.loc[remaining]
                  .sort_values('log_prod_pc', ascending=False).index.tolist())
    for i, cid in enumerate(rem_sorted):
        name_map[cid] = 'Diversified Producers' if i == 0 else f'Mixed Producers {i}'

centroids_df['ClusterLabels'] = centroids_df.index.map(name_map)
snap_fit['ClusterLabels'] = snap_fit['cluster_id'].map(name_map)

print('Assigned names:')
print(centroids_df[['n', 'dominant_rent', 'log_prod_pc', 'ClusterLabels']].to_string())


### Save assignments and update the panel

Overwrite `clusters_1995.csv` and patch `panel.parquet`'s `ClusterLabels`
so notebooks 3, 4, 5 and every viz notebook see the new labels.


In [ ]:
out_clusters = snap_fit[['Country Code', 'Country Name', 'ClusterLabels']].copy()
out_clusters.columns = ['Country Code', 'Country', 'ClusterLabels']
out_clusters = out_clusters.drop_duplicates('Country Code')
out_clusters.to_csv(artifact_path('clusters_1995.csv'), index=False)
print(f'  Saved clusters_1995.csv ({len(out_clusters)} countries)')

code_to_label = dict(zip(out_clusters['Country Code'], out_clusters['ClusterLabels']))
panel = panel.drop(columns=['ClusterLabels'], errors='ignore')
panel['ClusterLabels'] = panel['Country Code'].map(code_to_label)
panel.to_parquet(artifact_path('panel.parquet'), index=False)
print('  Updated panel.parquet with new ClusterLabels')

print('\nCountries per cluster:')
for label in out_clusters['ClusterLabels'].unique():
    members = out_clusters.loc[out_clusters['ClusterLabels'] == label, 'Country'].tolist()
    print(f'  {label} ({len(members)}): {", ".join(members)}')


## Rent breakdown by cluster (% of GDP)

For each cluster, average the four rent components and the total NR rent
share. Two views:

- **Snapshot (1995 only)** — what the clustering actually saw
- **Panel (1995–2019)** — long-run average composition per cluster

Saved as `cluster_rent_breakdown.csv`.


In [ ]:
snap_view = (snap_fit
    .groupby('ClusterLabels')[RENT_COLS + ['log_prod_pc']]
    .mean()
    .copy())
snap_view['Total NR rents (% of GDP)'] = snap_view[RENT_COLS].sum(axis=1)
snap_view['n_countries'] = snap_fit.groupby('ClusterLabels').size()

snap_view = snap_view[
    ['n_countries'] + RENT_COLS + ['Total NR rents (% of GDP)', 'log_prod_pc']
]
snap_view.columns = ['N', 'Oil', 'Gas', 'Mineral', 'Forestry',
                     'Total NR', 'log Prod p.c.']

print('Mean rent shares by cluster, 1995 (% of GDP):')
print(snap_view.round(2).to_string())
print()

panel_rent_cols = [c for c in RENT_COLS if c in panel.columns]
panel_view_src = panel.copy()
for c in panel_rent_cols:
    panel_view_src[c] = panel_view_src[c].fillna(0)
panel_view = (panel_view_src.dropna(subset=['ClusterLabels'])
    .groupby('ClusterLabels')[panel_rent_cols]
    .mean())
panel_view['Total NR'] = panel_view.sum(axis=1)
panel_view.columns = ['Oil', 'Gas', 'Mineral', 'Forestry', 'Total NR']

print('Mean rent shares by cluster, 1995–2019 panel (% of GDP):')
print(panel_view.round(2).to_string())


In [ ]:
out = snap_view.copy()
out.columns = ['N', 'Oil_1995', 'Gas_1995', 'Mineral_1995', 'Forestry_1995',
               'TotalNR_1995', 'logProdPc_1995']
for src_col, dst_col in [('Oil', 'Oil_panel'), ('Gas', 'Gas_panel'),
                          ('Mineral', 'Mineral_panel'),
                          ('Forestry', 'Forestry_panel'),
                          ('Total NR', 'TotalNR_panel')]:
    out[dst_col] = panel_view[src_col]

out = out.reset_index().rename(columns={'ClusterLabels': 'Cluster'})
out.to_csv(artifact_path('cluster_rent_breakdown.csv'), index=False)
print(f'  Saved cluster_rent_breakdown.csv ({len(out)} clusters)')


## Pearson correlations with ECI

For each feature in `CAT_MAP`, compute Pearson r against ECI on the panel.
Skip features that are missing from `Master.csv` or have fewer than 20
non-null observations.


In [ ]:
rows = []
for col, (short, cat) in CAT_MAP.items():
    if col not in panel.columns:
        continue
    sub = panel[[col, ECI_COL]].dropna()
    if len(sub) < 20:
        continue
    r = float(np.corrcoef(sub[col], sub[ECI_COL])[0, 1])
    rows.append(dict(feature=col, short=short, cat=cat, r=r))

corr_df = pd.DataFrame(rows).sort_values('r').reset_index(drop=True)
corr_df.to_csv(artifact_path('corr_with_eci.csv'), index=False)
print(f'  Saved corr_with_eci.csv ({len(corr_df)} features)')


## PCA on 1995 cross-section

Wider macro feature set than the clustering used (rents + macro structure
variables). PC1/PC2 saved as a 2D country positioning that the viz
notebook overlays cluster colours onto.


In [ ]:
PCA_FEATS = [
    'Total_Production_Value_Per_Capita',
    'Total natural resources rents (% of GDP)',
    'Oil rents (% of GDP)',
    'Natural gas rents (% of GDP)',
    'Mineral rents (% of GDP)',
    'Forestry rents (% of GDP)',
    'GDP per capita (constant prices, PPP)',
    'Trade (% of GDP)',
    'Manufacturing',
    'Services',
]
pca_feats_present = [c for c in PCA_FEATS if c in panel.columns]

snap_pca = panel[panel['Year'] == 1995].copy()
snap_pca = snap_pca.dropna(subset=pca_feats_present + ['ClusterLabels'])

X = StandardScaler().fit_transform(snap_pca[pca_feats_present].values)
pca = PCA(n_components=2, random_state=42).fit(X)
PCs = pca.transform(X)

scatter_df = pd.DataFrame({
    'Country Code': snap_pca['Country Code'].values,
    'Country':      snap_pca['Country Name'].values,
    'ClusterLabels': snap_pca['ClusterLabels'].values,
    'PC1':          PCs[:, 0],
    'PC2':          PCs[:, 1],
})
scatter_df.to_csv(artifact_path('pca_scatter.csv'), index=False)
print(f'  Saved pca_scatter.csv ({len(scatter_df)} countries)')
print(f'  PC1 = {pca.explained_variance_ratio_[0]*100:.1f}%, '
      f'PC2 = {pca.explained_variance_ratio_[1]*100:.1f}%, '
      f'total = {sum(pca.explained_variance_ratio_)*100:.1f}%')

loadings_df = pd.DataFrame({
    'feature': pca_feats_present,
    'short':   [NAME_MAP.get(c, c) for c in pca_feats_present],
    'PC1':     pca.components_[0],
    'PC2':     pca.components_[1],
    'PC1_var': pca.explained_variance_ratio_[0],
    'PC2_var': pca.explained_variance_ratio_[1],
})
loadings_df.to_csv(artifact_path('pca_loadings.csv'), index=False)
print(f'  Saved pca_loadings.csv ({len(loadings_df)} features)')


## Median ECI by cluster, over time

Pre-aggregate median ECI plus standard error per (year, cluster). Uses
the new ClusterLabels written above.


In [ ]:
traj = (
    panel.dropna(subset=['ClusterLabels', ECI_COL])
         .groupby(['Year', 'ClusterLabels'])[ECI_COL]
         .agg(['median', 'std', 'count'])
         .reset_index()
)
traj.columns = ['Year', 'ClusterLabels', 'median', 'std', 'n']
traj['se'] = traj['std'] / np.sqrt(traj['n'])
traj.to_csv(artifact_path('median_eci_by_cluster.csv'), index=False)
print(f'  Saved median_eci_by_cluster.csv ({len(traj)} rows)')


## Summary

| Artifact | Rows | Used by |
|---|---|---|
| clusters_1995.csv | one per country | charts 5, 6, 26 (overwritten here) |
| cluster_rent_breakdown.csv | 5 | new diagnostic table |
| corr_with_eci.csv | one per feature | chart 4 |
| pca_scatter.csv | one per country (1995) | chart 26 |
| pca_loadings.csv | one per feature | charts 27, 28 |
| median_eci_by_cluster.csv | one per (year, cluster) | chart 6 |

`panel.parquet` is also re-saved with the new `ClusterLabels` column.
Notebooks 3, 4, 5 and the viz notebooks pick up the updated labels
automatically when they re-read the parquet.

If the cluster names still look wrong after running this, check:

1. The centroid table — does `dominant_rent` match the assigned label?
2. The country lists — are obvious archetypes (Saudi Arabia, Chile, DRC)
   in the cluster you'd expect?
3. The rent-breakdown table — is the dominant rent component large
   relative to the others?

If any answer is no, the clustering features themselves
(`RENT_COLS + ['log_prod_pc']` at the top of the K-means cell) are the
place to change.
